# Analytics & Visualization — TechJobAI

Sinh tất cả biểu đồ báo cáo từ `data/it_jobs_processed.csv` và các model đã train.

**Đầu ra** (vào `reports/figures/`):

| File | Nội dung |
|------|----------|
| `eda_overview.png` | Phân phối lương, seniority, domain, state |
| `salary_model_results.png` | Actual vs Predicted + Residuals |
| `13_demand_radar.png` | Demand score theo domain |
| `08_posting_frequency.png` | Tần suất đăng tuyển theo domain |
| `elbow_method.png` | Elbow method chọn K cho KMeans |
| `cluster_results.png` | PCA scatter plot các cụm |
| `top_skills.png` | Top 15 kỹ năng IT |
| `salary_distribution.png` | Salary distribution by seniority |
| `top_locations.png` | Top 15 bang tuyển dụng nhiều |
| `hiring_trend.png` | Hiring trend by domain |

In [ ]:
import os, json, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_palette('pastel')

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_FILE = os.path.join(BASE_DIR, 'data', 'it_jobs_processed.csv')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
FIGURES_DIR = os.path.join(BASE_DIR, 'reports', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

df = pd.read_csv(DATA_FILE)
print(f'Dataset: {len(df):,} rows, {df.shape[1]} cols')

# Load models
salary_model = joblib.load(os.path.join(MODELS_DIR, 'best_salary_model.joblib'))
salary_meta = joblib.load(os.path.join(MODELS_DIR, 'salary_model_meta.joblib'))
demand_meta = joblib.load(os.path.join(MODELS_DIR, 'demand_meta.joblib'))
cluster_meta = joblib.load(os.path.join(MODELS_DIR, 'cluster_meta.joblib'))

figure_meta = []
def save_fig(name, title):
    plt.savefig(os.path.join(FIGURES_DIR, name), dpi=150, bbox_inches='tight')
    plt.close()
    figure_meta.append({'filename': name, 'title': title, 'created': datetime.now().isoformat()})
    print(f'  Saved {name}')

In [ ]:
# 1. EDA Overview — 4 biểu đồ
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sal = df['salary_annual'].dropna()
axes[0,0].hist(sal, bins=50, color='skyblue', edgecolor='white')
axes[0,0].set_title(f'Salary Distribution (n={len(sal):,})')
axes[0,0].set_xlabel('Annual Salary ($)')

df['seniority_level'].value_counts().plot(kind='bar', ax=axes[0,1], color=['#0d6efd','#198754','#ffc107','#dc3545'])
axes[0,1].set_title('Seniority Level Distribution')

df['it_domain'].value_counts().plot(kind='bar', ax=axes[1,0], color='#0d6efd')
axes[1,0].set_title('IT Domain Distribution')

df['state'].value_counts().head(10).plot(kind='bar', ax=axes[1,1], color='#20c997')
axes[1,1].set_title('Top 10 States')

plt.tight_layout()
save_fig('eda_overview.png', 'EDA Overview')

In [ ]:
# 2. Salary Model Results
features = salary_meta['feature_names']
numeric_features = salary_meta['numeric_features']
categorical_features = salary_meta['categorical_features']

salary_df = df.dropna(subset=['salary_annual']).copy()
Q1, Q3 = salary_df['salary_annual'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lo, hi = max(Q1 - 1.5*IQR, 15000), min(Q3 + 1.5*IQR, 500000)
salary_df = salary_df[(salary_df['salary_annual'] >= lo) & (salary_df['salary_annual'] <= hi)]
X = salary_df[features]
y = salary_df['salary_annual']

y_pred = salary_model.predict(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y, y_pred, alpha=0.3, s=10, c='skyblue')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual ($)')
axes[0].set_ylabel('Predicted ($)')
axes[0].set_title(f'Actual vs Predicted (R²={salary_meta["r2_score"]:.3f})')

residuals = y - y_pred
axes[1].hist(residuals, bins=50, color='#0d6efd', edgecolor='white')
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residual Distribution (MAE=${salary_meta["mae"]:,.0f})')

plt.tight_layout()
save_fig('salary_model_results.png', 'Salary Model Evaluation')

In [ ]:
# 3. Demand Radar + Posting Frequency
demand_df = df.groupby('it_domain').size().reset_index(name='count')
demand_df['score'] = np.log1p(demand_df['count']) / np.log1p(demand_df['count']).max() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(demand_df['it_domain'], demand_df['score'], color='#ffc107')
axes[0].set_title('Demand Score by Domain')
axes[0].set_xlabel('Demand Score (0-100)')

axes[1].barh(demand_df['it_domain'], demand_df['count'], color='#0d6efd')
axes[1].set_title('Posting Frequency by Domain')
axes[1].set_xlabel('Number of Postings')

plt.tight_layout()
save_fig('13_demand_radar.png', 'Demand Score by Domain')
save_fig('08_posting_frequency.png', 'Posting Frequency by Domain')

In [ ]:
# 4. Elbow Method + Cluster Results
cluster_model = joblib.load(os.path.join(MODELS_DIR, 'cluster_model.joblib'))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Elbow
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
cluster_df = df.dropna(subset=['salary_annual']).copy()
X_cl = cluster_df[features]
pre = ColumnTransformer([('num', StandardScaler(), numeric_features), ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])
X_t = pre.fit_transform(X_cl)
if X_t.shape[0] > 5000:
    idx = np.random.RandomState(42).choice(X_t.shape[0], 5000, replace=False)
    X_t = X_t[idx]
inertias = [KMeans(n_clusters=k, random_state=42, n_init=5).fit(X_t).inertia_ for k in range(2, 11)]
axes[0].plot(range(2, 11), inertias, marker='o', color='lightgreen')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')

# Cluster scatter
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_t)
labels = KMeans(n_clusters=5, random_state=42, n_init=5).fit_predict(X_t)
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='Set1', alpha=0.5, s=15)
axes[1].set_title('Cluster Visualization (PCA)')
axes[1].legend(*scatter.legend_elements(), title='Cluster')

plt.tight_layout()
save_fig('elbow_method.png', 'Elbow Method')
save_fig('cluster_results.png', 'Cluster Results')

In [ ]:
# 5. Extra figures
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Top skills
skill_cols = [c for c in df.columns if c.startswith('skill_') and c != 'skill_soft_skills']
skill_sums = df[skill_cols].sum().sort_values(ascending=False).head(15)
axes[0,0].barh(range(len(skill_sums)), skill_sums.values, color='#0d6efd')
axes[0,0].set_yticks(range(len(skill_sums)))
axes[0,0].set_yticklabels(skill_sums.index, fontsize=8)
axes[0,0].set_title('Top 15 Skills')

# Salary by seniority
sal_df = df.dropna(subset=['salary_annual', 'seniority_level'])
sns.boxplot(data=sal_df, x='seniority_level', y='salary_annual', ax=axes[0,1],
            palette=['#0d6efd','#198754','#ffc107','#dc3545'],
            order=['Junior', 'Mid', 'Senior', 'Manager'])
axes[0,1].set_title('Salary Distribution by Seniority')

# Top locations
df['state'].value_counts().head(15).plot(kind='bar', ax=axes[1,0], color='#20c997')
axes[1,0].set_title('Top 15 Hiring States')

# Hiring trend by domain
domain_counts = df['it_domain'].value_counts()
axes[1,1].pie(domain_counts.values, labels=domain_counts.index, autopct='%1.1f%%', startangle=90)
axes[1,1].set_title('Job Distribution by Domain')

plt.tight_layout()
save_fig('top_skills.png', 'Top Skills')
save_fig('salary_distribution.png', 'Salary by Seniority')
save_fig('top_locations.png', 'Top Locations')

## Lưu metadata

Ghi danh sách figures đã tạo vào `figure_metadata.json`.

In [ ]:
with open(os.path.join(FIGURES_DIR, 'figure_metadata.json'), 'w') as f:
    json.dump(figure_meta, f, indent=2)
print(f'Saved {len(figure_meta)} figure metadata entries')
print(f'\nAll figures in: {FIGURES_DIR}')